[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madhudream/rag_simple/blob/main/colab/04_query_transforms.ipynb)

# 04 — Query transforms: fix the pool by fixing the question

Companion notebook to blog post **04 (Query transforms)**. Post 03 proved a reranker
can't fetch what retrieval missed. This notebook changes the QUESTION instead — and
flips a wrong answer ("8 pm") to the right one ("5 pm").

The tourist-and-phrasebook metaphor:
1. Retrieval matches **phrasing, not intent** ("Saturday" query vs "weekends" doc)
2. **Rephrasing ≠ translating** — same-dialect rewrites retrieve the same wrong docs
3. **Ask several ways** in the corpus's dialect, pool the lists (multi-query + RRF)
4. Or **sketch the answer** and search with it (HyDE) — invented specifics included

Needs an OpenAI API key (rewrites + answers). LLM rewrites vary run to run — your exact
phrasings will differ; the pattern (translated rewrite → doc enters pool → answer flips)
is what reproduces.

In [ ]:
%pip install -q fastembed numpy openai

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## Setup — corpus and the bi-encoder retriever (posts 2b/03)

In [ ]:
import numpy as np
from fastembed import TextEmbedding
from openai import OpenAI

corpus = [
    "Grooming appointments must be booked at least 48 hours in advance",           # 0
    "The boarding facility closes at 7 pm on weekdays and 5 pm on weekends",       # 1
    "Dogs staying longer than three nights receive a complimentary bath before pickup",  # 2
    "Refunds for cancelled boarding are issued within 5 business days",            # 3
    "Bookings made for public holidays are non-refundable",                        # 4
    "Refunds for cancelled grooming appointments are issued within 10 business days",    # 5
    "All pets must have up to date rabies vaccination records on file",            # 6
    "Daycare drop off starts at 6:30 am and the last pickup is at 8 pm",           # 7
    "A late pickup fee of 15 dollars applies for every 30 minutes after closing",  # 8
    "Error E-4042 refund transaction declined by the payment gateway",             # 9
    "Error E-4043 refund transaction succeeded but receipt email failed",          # 10
    "Error E-4044 refund transaction pending manual review",                       # 11
]

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
doc_embs = list(emb_model.embed(corpus))
client = OpenAI()

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def vector_search(query, top_k=3):
    q = list(emb_model.embed([query]))[0]
    return [i for s, i in sorted(((cosine(q, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:top_k]]

def llm(prompt):
    r = client.chat.completions.create(model="gpt-5.4-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

## The dialect gap — the failure reranking couldn't save

Question says "Saturday", the answer doc says "weekends" → dense rank 5, outside any
small pool. Feed the top-3 pool to the LLM and get a confident WRONG answer.

In [ ]:
Q = "Until what time can I pick up my dog on a Saturday?"

qe = list(emb_model.embed([Q]))[0]
print("full dense ranking:")
for rank, (s, i) in enumerate(sorted(((cosine(qe, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:6], 1):
    mark = "  <-- right doc (weekends = Saturday)" if i == 1 else ""
    print(f"  rank {rank}  {s:.3f}  doc {i}: {corpus[i][:58]}{mark}")

PROMPT = """Answer the question using ONLY the context below.

Context:
{context}

Question: {question}
Answer:"""

def answer_with(ids, question):
    ctx = "\n".join(f"[doc {i}] {corpus[i]}" for i in ids)
    return llm(PROMPT.format(context=ctx, question=question))

baseline_pool = vector_search(Q, 3)
print("\nbaseline pool:", baseline_pool)
print("LLM answer   :", answer_with(baseline_pool, Q))

## First attempt — naive paraphrasing (watch it fail)

Generic "rewrite 3 ways" keeps the user's vocabulary: every rewrite still says
dog/Saturday/pickup → same embedding neighborhood → same wrong docs, four times.

In [ ]:
naive = llm(f"Rewrite the following question 3 different ways for searching a pet care "
            f"center's policy handbook. Use different vocabulary each time. "
            f"Return one rewrite per line, no numbering.\n\nQuestion: {Q}")
naive_rewrites = [l.strip() for l in naive.splitlines() if l.strip()]

for rw in naive_rewrites:
    print(" -", rw)
print("\nretrieval per query (top-3 ids):")
for qq in [Q] + naive_rewrites:
    print(f"  {vector_search(qq, 3)}  <- {qq[:65]}")

## Multi-query done right — translate into the corpus's dialect

The fix is the PROMPT: tell the LLM what dialect the handbook speaks (categories, not
specifics). Watch a rewrite say "weekend" — and walk the missing doc into its top-3.

In [ ]:
REWRITE_PROMPT = """You are helping search a pet care center's policy handbook.
Handbooks use general policy language: categories instead of specifics
(a specific day becomes "weekday" or "weekend", a pet becomes the service area
like "boarding" or "daycare", times become "closing time" or "opening hours").

Rewrite the question below 3 different ways as the HANDBOOK would phrase the
underlying policy. Generalize the specifics. One rewrite per line, no numbering.

Question: {q}"""

rewrites = [l.strip() for l in llm(REWRITE_PROMPT.format(q=Q)).splitlines() if l.strip()]
for rw in rewrites:
    print(" -", rw)

lists = [vector_search(qq, 3) for qq in [Q] + rewrites]
print("\nretrieval per query (top-3 ids):")
for qq, ids in zip([Q] + rewrites, lists):
    mark = "  <-- doc 1 walked in!" if 1 in ids else ""
    print(f"  {ids}{mark}  <- {qq[:60]}")

## Fuse the lists — RRF from post 2c, verbatim

Merging ranked lists by position is the same problem whether the lists come from two
search engines (2c) or four phrasings. Then the finale: pool as context → answer flips.

In [ ]:
from collections import Counter

def rrf_fuse(rankings, k=60, top=5):
    points = Counter()
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            points[doc_id] += 1 / (k + rank)
    return [i for i, _ in points.most_common(top)]

fused_pool = rrf_fuse(lists)
print("baseline pool:", baseline_pool, "        (doc 1 in pool:", 1 in baseline_pool, ")")
print("fused pool   :", fused_pool, " (doc 1 in pool:", 1 in fused_pool, ")")

print("\nLLM answer with baseline pool:", answer_with(baseline_pool, Q))
print("LLM answer with fused pool   :", answer_with(fused_pool, Q))

Note what flipped it: the EMBEDDING never learned Saturday = weekend (the right doc
is still last in the pool). But the generating LLM connects them instantly once the doc
is in the room. Retrieval's job was never rank 1 — it was getting it in the room.

## HyDE — search with a fake answer

Write a hypothetical handbook sentence, embed THE FAKE, search with its vector
(answers live near answers on the map). Study the output: perfect handbook STYLE,
invented FACTS — there is no 6 p.m. rule. That invention is HyDE's danger: leak it
into generation and faithfulness pays (course: 0.909 → 0.815).

In [ ]:
fake = llm(f"Write one sentence that could appear in a pet care center's policy "
           f"handbook and would answer this question. Invent plausible specifics."
           f"\n\nQuestion: {Q}")
print("fake doc:", fake)

fe = list(emb_model.embed([fake]))[0]
print("\nsearch with the FAKE's embedding:")
for rank, (s, i) in enumerate(sorted(((cosine(fe, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:3], 1):
    mark = "  <-- right doc" if i == 1 else ""
    print(f"  rank {rank}  {s:.3f}  doc {i}: {corpus[i][:58]}{mark}")

## The bill — an LLM call per query, forever

In [ ]:
import time

t0 = time.perf_counter(); vector_search(Q, 3); t_base = time.perf_counter() - t0
t0 = time.perf_counter()
rws = [l.strip() for l in llm(REWRITE_PROMPT.format(q=Q)).splitlines() if l.strip()]
for x in [Q] + rws:
    vector_search(x, 3)
t_multi = time.perf_counter() - t0

print(f"baseline   : 1 retrieval                  = {t_base*1000:6.0f} ms")
print(f"multi-query: 1 LLM call + 4 retrievals    = {t_multi*1000:6.0f} ms")

## PRODUCTION — LlamaIndex `QueryFusionRetriever`

Generate N rewrites, retrieve for each, RRF-fuse — one object. Course-scale results
(full corpus, 50 golden Qs, on top of hybrid + rerank): recall 0.73 → 0.78 (pool finally
moved after two flat posts — post 03's diagnosis proven), HyDE same recall but
faithfulness 0.909 → 0.815 (fake-answer leak), latency 2.56 s → 6.50 s (the bill).

In [ ]:
%pip install -q llama-index

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = LlamaOpenAI(model="gpt-5.4-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

docs = [Document(text=t, metadata={"doc_id": f"doc {i}"}) for i, t in enumerate(corpus)]
index = VectorStoreIndex.from_documents(docs)

retriever = QueryFusionRetriever(
    [index.as_retriever(similarity_top_k=3)],
    llm=Settings.llm,
    num_queries=4,                    # 1 original + 3 generated rewrites
    mode="reciprocal_rerank",         # RRF — post 2c's formula, again
    similarity_top_k=5,
)
for n in retriever.retrieve(Q):
    mark = "  <-- in the pool" if n.node.metadata["doc_id"] == "doc 1" else ""
    print(f"  {n.score:.4f}  {n.node.metadata['doc_id']}: {n.node.get_content()[:58]}{mark}")

## Recap — the phrasebook's rules

1. **Retrieval matches phrasing, not intent** → the dialect gap ("Saturday" vs "weekends")
2. **Rephrasing ≠ translating** → naive paraphrases retrieved the same wrong docs 4×
3. **Ask several ways, pool the lists** → dialect rewrites + RRF: doc entered the pool,
   answer flipped 8 pm → 5 pm
4. **Or sketch the answer (HyDE)** → style-matched fakes as search keys; invented
   specifics cost faithfulness

**Next: contextual retrieval (05)** — close the same vocabulary gap at INDEX time, once
per document instead of once per query. It made multi-query obsolete in the course.